# STAGE 4: Ensemble & Optimization
## Goal: Combine best models for maximum performance

This notebook covers:
1. Hyperparameter tuning on best model (XGBoost)
2. Building soft voting ensemble
3. Comparing optimized model vs ensemble
4. Final selection between two candidates


## Section 1: Import and Load Models from Stage 3

In [7]:
import pandas as pd
import numpy as np
import pickle
import json
from sklearn.ensemble import VotingClassifier
from sklearn.model_selection import GridSearchCV
import xgboost as xgb
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

# Load data
stage1_data = pickle.load(open('../Stage_1_Data_Prep/stage1_preprocessed_data.pkl', 'rb'))
X_train_scaled = stage1_data['X_train_scaled']
X_test_scaled = stage1_data['X_test_scaled']
y_train = stage1_data['y_train']
y_test = stage1_data['y_test']

# Load models from Stage 3
gb = pickle.load(open('../Stage_3_Accuracy_Models/stage3_gradient_boosting_model.pkl', 'rb'))
xgb_original = pickle.load(open('../Stage_3_Accuracy_Models/stage3_xgboost_model.pkl', 'rb'))
mlp = pickle.load(open('../Stage_3_Accuracy_Models/stage3_mlp_model.pkl', 'rb'))
rf = pickle.load(open('../Stage_2_Fast_Models/stage2_random_forest_model.pkl', 'rb'))

# Load stage 3 results
stage3_results = json.load(open('../Stage_3_Accuracy_Models/stage3_results.json', 'r'))
best_stage3_model = stage3_results['best_model']

print(f"Loaded all models from Stage 3")
print(f"  Best model from Stage 3: {best_stage3_model}")

Loaded all models from Stage 3
  Best model from Stage 3: Gradient Boosting


## Section 2: Hyperparameter Tuning for Best Model

Tune the best model from Stage 3 (detects if it was Gradient Boosting, XGBoost, or MLP) and create ensemble.


In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.base import clone

print(f"Hyperparameter tuning for {best_stage3_model}...\n")

# Select model and parameters based on best model from Stage 3
if best_stage3_model == "Gradient Boosting":
    print(f"Optimizing Gradient Boosting (best model from Stage 3)\n")
    
    param_grid = {
        'n_estimators': [100, 150],
        'learning_rate': [0.1, 0.15],
        'max_depth': [5, 6]
    }
    
    grid_search = GridSearchCV(
        GradientBoostingClassifier(random_state=42),
        param_grid,
        cv=3,
        scoring='f1',
        n_jobs=-1,
        verbose=0
    )
    best_model_to_tune = gb

elif best_stage3_model == "XGBoost":
    print(f"Optimizing XGBoost (best model from Stage 3)\n")
    
    scale_pos_weight = sum(y_train == 0) / sum(y_train == 1)
    
    param_grid = {
        'max_depth': [5, 6],
        'learning_rate': [0.1, 0.15],
        'n_estimators': [150, 200]
    }
    
    grid_search = GridSearchCV(
        xgb.XGBClassifier(scale_pos_weight=scale_pos_weight, random_state=42),
        param_grid,
        cv=3,
        scoring='f1',
        n_jobs=-1,
        verbose=0
    )
    best_model_to_tune = xgb_original

else:
    print(f"Optimizing {best_stage3_model}...\n")
    best_model_to_tune = mlp

print("Grid searching (this may take a few minutes)...")
grid_search.fit(X_train_scaled, y_train)
model_optimized = grid_search.best_estimator_

print(f"\nGrid search completed")
print(f"  Best parameters: {grid_search.best_params_}")
print(f"  Best CV F1-score: {grid_search.best_score_:.4f}")

Hyperparameter tuning for Gradient Boosting...

Optimizing Gradient Boosting (best model from Stage 3)

Grid searching (this may take a few minutes)...


## Section 3: Evaluate Optimized Model

In [ ]:
# Get predictions from optimized model
y_prob_opt = model_optimized.predict_proba(X_test_scaled)[:, 1]
y_pred_opt = model_optimized.predict(X_test_scaled)

metrics_opt = {
    'Model': f'{best_stage3_model} (Optimized)',
    'F1-Score': f1_score(y_test, y_pred_opt),
    'Precision': precision_score(y_test, y_pred_opt),
    'Recall': recall_score(y_test, y_pred_opt),
    'ROC-AUC': roc_auc_score(y_test, y_prob_opt)
}

print(f"\n{best_stage3_model} Optimized Results:")
for key, value in metrics_opt.items():
    if key != 'Model':
        print(f"  {key}: {value:.4f}")


XGBoost Optimized Results:
  F1-Score: 0.9999
  Precision: 0.9999
  Recall: 0.9999
  ROC-AUC: 1.0000


## Section 4: Build Soft Voting Ensemble

Combine best model (optimized) with other top performers from Stage 3


In [ ]:
print("Building Voting Ensemble...\n")
print(f"Ensemble includes: {best_stage3_model} (optimized), Gradient Boosting, MLP\n")

# Prepare ensemble estimators - use the optimized best model
ensemble_estimators = []
if best_stage3_model == "Gradient Boosting":
    ensemble_estimators = [
        ('gb_opt', model_optimized),
        ('xgb', xgb_original),
        ('mlp', mlp)
    ]
    weights = [1.5, 1.2, 1.0]
elif best_stage3_model == "XGBoost":
    ensemble_estimators = [
        ('xgb_opt', model_optimized),
        ('gb', gb),
        ('mlp', mlp)
    ]
    weights = [1.5, 1.2, 1.0]
else:
    ensemble_estimators = [
        ('mlp_opt', model_optimized),
        ('gb', gb),
        ('xgb', xgb_original)
    ]
    weights = [1.5, 1.2, 1.0]

# Create ensemble with weighted voting
voting_ensemble = VotingClassifier(
    estimators=ensemble_estimators,
    voting='soft',  # Average probability scores
    weights=weights  # Weight best models more
)

# Fit on training data
voting_ensemble.fit(X_train_scaled, y_train)

# Get predictions
y_prob_ensemble = voting_ensemble.predict_proba(X_test_scaled)[:, 1]
y_pred_ensemble = voting_ensemble.predict(X_test_scaled)

metrics_ensemble = {
    'Model': 'Voting Ensemble',
    'F1-Score': f1_score(y_test, y_pred_ensemble),
    'Precision': precision_score(y_test, y_pred_ensemble),
    'Recall': recall_score(y_test, y_pred_ensemble),
    'ROC-AUC': roc_auc_score(y_test, y_prob_ensemble)
}

print(f"Ensemble created and evaluated")
for key, value in metrics_ensemble.items():
    if key != 'Model':
        print(f"  {key}: {value:.4f}")

Building Voting Ensemble...



KeyboardInterrupt: 

## Section 5: Final Stage 4 Comparison

In [ ]:
# Load original best model predictions from Stage 3
stage3_preds = pickle.load(open('../Stage_3_Accuracy_Models/stage3_predictions.pkl', 'rb'))

# Get metrics for original best model from Stage 3
if best_stage3_model == "Gradient Boosting":
    y_prob_original = stage3_preds['y_prob_gb']
elif best_stage3_model == "XGBoost":
    y_prob_original = stage3_preds['y_prob_xgb']
else:
    y_prob_original = stage3_preds['y_prob_mlp']

y_pred_original = (y_prob_original > 0.5).astype(int)

metrics_original = {
    'Model': f'{best_stage3_model} (Original from Stage 3)',
    'F1-Score': f1_score(y_test, y_pred_original),
    'Precision': precision_score(y_test, y_pred_original),
    'Recall': recall_score(y_test, y_pred_original),
    'ROC-AUC': roc_auc_score(y_test, y_prob_original)
}

# Create comparison
stage4_comparison = pd.DataFrame([metrics_original, metrics_opt, metrics_ensemble])
stage4_comparison_sorted = stage4_comparison.sort_values('F1-Score', ascending=False)

print("="*80)
print("STAGE 4: ENSEMBLE & OPTIMIZATION RESULTS")
print("="*80)
print(stage4_comparison_sorted.to_string(index=False))
print("="*80)

stage4_comparison_sorted.to_csv('stage4_comparison.csv', index=False)

## Section 6: Select Final Candidate for Stage 5

Choose between: Optimized XGBoost or Ensemble

In [ ]:
# Determine best candidate
best_f1 = stage4_comparison_sorted.iloc[0]['F1-Score']
best_model_stage4 = stage4_comparison_sorted.iloc[0]['Model']

print("="*80)
print("STAGE 4: FINAL DECISION")
print("="*80)
print(f"\nBest Performer: {best_model_stage4}")
print(f"F1-Score: {best_f1:.4f}")
print(f"\nImprovement from tuning:")
f1_improvement = (metrics_opt['F1-Score'] - metrics_original['F1-Score']) * 100
print(f"  {best_stage3_model} original → optimized: +{f1_improvement:.2f}% F1")
print(f"\nEnsemble benefit:")
ensemble_improvement = (metrics_ensemble['F1-Score'] - metrics_opt['F1-Score']) * 100
print(f"  Optimized {best_stage3_model} → Ensemble: +{ensemble_improvement:.2f}% F1")

print(f"\n RECOMMENDATION FOR STAGE 5:")
if ensemble_improvement > 0.5:
    print(f"  Proceed with: ENSEMBLE (better accuracy)")
    final_model_stage4 = voting_ensemble
    final_model_name = "Voting Ensemble"
else:
    print(f"  Proceed with: {best_stage3_model} Optimized (simpler, similar accuracy)")
    final_model_stage4 = model_optimized
    final_model_name = f"{best_stage3_model} Optimized"

## Section 7: Save Stage 4 Results

In [ ]:
# Save optimized models
pickle.dump(model_optimized, open(f'stage4_{best_stage3_model.lower().replace(" ", "_")}_optimized.pkl', 'wb'))
pickle.dump(voting_ensemble, open('stage4_voting_ensemble.pkl', 'wb'))

# Save predictions
stage4_predictions = {
    'y_prob_opt': y_prob_opt,
    'y_prob_ensemble': y_prob_ensemble
}
pickle.dump(stage4_predictions, open('stage4_predictions.pkl', 'wb'))

# Save metrics and grid search results
stage4_results = {
    'best_stage3_model': best_stage3_model,
    'best_model': best_model_stage4,
    'final_model_for_stage5': final_model_name,
    'best_f1': float(best_f1),
    'best_params': grid_search.best_params_,
    'best_cv_score': float(grid_search.best_score_)
}
json.dump(stage4_results, open('stage4_results.json', 'w'), indent=2)

print("Results saved:")
print(f"  - stage4_{best_stage3_model.lower().replace(' ', '_')}_optimized.pkl")
print(f"  - stage4_voting_ensemble.pkl")
print(f"  - stage4_predictions.pkl")
print(f"  - stage4_comparison.csv")
print(f"  - stage4_results.json")

## Summary
Key Achievements:
1. Hyperparameter tuned XGBoost
2. Built voting ensemble from 3 best models
3. Compared optimized model vs ensemble
4. Selected best candidate for final validation

**Next Step:** Proceed to **Stage 5: Final Validation & Winner**
- Perform rigorous 5-fold cross-validation
- Generate final comparison table
- Declare official winner
- Create experiment summary report